# Shoreline on Kaggle

Train a small model to answer **Land or Water?** from a latitude and longitude.

Upload this repository as a Kaggle dataset, attach it to the notebook, and turn on a GPU. A T4 is enough for the default 0.5B model.


In [ ]:
from pathlib import Path
import sys

def find_repo() -> Path:
    here = Path.cwd()
    candidates = [here, *here.parents]
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates.extend(kaggle.rglob('openenv.yaml'))
    for candidate in candidates:
        root = candidate if candidate.is_dir() else candidate.parent
        if (root / 'shoreline' / 'data' / 'land_mask.npz').is_file():
            return root
    raise FileNotFoundError('Add the Shoreline dataset to this notebook.')

ROOT = find_repo()
sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'peft', 'transformers', 'accelerate', 'huggingface_hub'])
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])


In [ ]:
from IPython.display import Image, display
from shoreline.sim import ShoreSim

sim = ShoreSim()
place = sim.reset(latitude=40.7, longitude=-74.0)
print(place.prompt)
display(Image(data=place.image_png))
print('reward for saying land:', sim.step('land').reward)

land_cells = float(sim.world.land.mean())
print(f'land cells: {land_cells:.1%}')
print(f'always answering water scores about {1 - land_cells:.1%} on a uniform sample')


## GRPO

Each step asks one coordinate four times. The four rewards become advantages inside that group, so a tied group does not move the model. Training samples land and water evenly. Judge the model afterwards on the full grid, not on this balanced reward.


In [ ]:
from shoreline.grpo import train_survey

history = train_survey(steps=20, group_size=4, save_steps=5, output_dir='/kaggle/working/shoreline' if Path('/kaggle/working').is_dir() else 'outputs/shoreline')
print('mean reward over the last 5 steps:', sum(history[-5:]) / len(history[-5:]))
